# C5.01 Corpus agenti - Tema 2


Scop: ne uităm la `data/corpus_typed.json`, verificăm distribuția bulelor și alegem ce exemple merită păstrate pentru vector store.



## 1. Setup

In [ ]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Alin\Desktop\ADC\An II\Sem II\AI Engineering\Proiect\echochamber-project-team-5")
os.chdir(PROJECT_ROOT)
DATA_PATH = Path("data/typed/corpus_axis_typed.jsonl")


In [6]:
DATA_PATH

WindowsPath('data/typed/corpus_axis_typed.jsonl')

## 2. Încărcăm corpusul

In [7]:
df = pd.read_json(DATA_PATH, lines=True)
df.head(2)

,id,source_channel,channel_family,video_title,text,target,stance,tone,institutional,legitimare,epistemic,geopolitic,mobilizare,justification,confidence,fallback,discourse_type,discourse_subtype,type_confidence
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...,georgescu,anti,acuzator,0,0,-1,0,0,Comentariul acuză pe Georgescu de intenții rel...,0.9,False,T3_opozitie_suveranista,opozitie_difuza,medium
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,georgesimionoficial,,#democratie #georgesimion #unitate #prosperita...,Mă voi realizați că votul s-a încheiat și Nicu...,nicusor_dan,pro,neutru,1,1,0,0,0,Comentariul afirmă că Nicușor Dan a fost ales ...,0.9,False,T5_pro_democratic_european,legitimitate_pluralista,medium


## 3. Vedem structura datelor

In [8]:
len(df)

420

In [9]:
print("Coloane:")
df.columns

Coloane:


Index(['id', 'source_channel', 'channel_family', 'video_title', 'text',
       'target', 'stance', 'tone', 'institutional', 'legitimare', 'epistemic',
       'geopolitic', 'mobilizare', 'justification', 'confidence', 'fallback',
       'discourse_type', 'discourse_subtype', 'type_confidence'],
      dtype='str')

In [10]:
df["discourse_type"].value_counts(dropna=False)

discourse_type
T6_afectiv_pozitional         402
T2_grievance_anti_sistem        7
T4_conspiratie_externalism      4
T5_pro_democratic_european      3
T1_suport_personalist           3
T3_opozitie_suveranista         1
Name: count, dtype: int64

In [26]:
## Exemplu de text pentru fiecare tip de discurs

In [11]:
for bubble in df["discourse_type"].value_counts().index:
    print("\n" + "="*80)
    print(bubble)
    print("="*80)
    
    sample = df[df["discourse_type"] == bubble]["text"].dropna().head(5)
    
    for i, text in enumerate(sample, 1):
        print(f"\n{i}. {text[:50]}")


T6_afectiv_pozitional

1. Lepra aia care ia interval este non stop lângă el 

2. Valoarea Președintelui ca om de stat, nu depinde d

3. De astazi am sters Realitatea TV si toate telkeviz

4. ce se va intampla acum cu judecătorii care au dezv

5. Craciun fericit, domnule Georgescu, alaturi de mil

T2_grievance_anti_sistem

1. La o facultate industrială nu vor să facă câteva i

2. Iar din nr acela 80 % dintre ei sau uitat la nea I

3. Si au pus labele spurcate pe țară in 89 securiștii

4. E foarte bine ca sunteți cei care cutremurați sist

5. zici ca suntem in filmul Mecanismul ... pentru cin

T4_conspiratie_externalism

1. Dictatorul Putin nu vrea pace, vrea capitularea ne

2. Votam masiv Nicusor Dan! Ai scris intr-o postare p

3. Deci apare agresiunea din partea Rusiei în general

4. Sufletul Pământului în rezervorul şi motorul başin

T5_pro_democratic_european

1. Mă voi realizați că votul s-a încheiat și Nicușor 

2. VICTORIE Ucraina Forte Zelenschi forte UE Euronato

3. Tipic sunt 

# 4. Pastram doar 150 de bula

In [12]:
# Alegem un subset simplu pentru curs: maximum 150 texte per bulă

N_PER_BUBBLE = 150

df_sample = (
    df[df["discourse_type"].notna()]
    .sort_values("type_confidence", ascending=False)
    .groupby("discourse_type", group_keys=False)
    .head(N_PER_BUBBLE)
    .copy()
)

df_sample["discourse_type"].value_counts()

discourse_type
T6_afectiv_pozitional         150
T2_grievance_anti_sistem        7
T4_conspiratie_externalism      4
T1_suport_personalist           3
T5_pro_democratic_european      3
T3_opozitie_suveranista         1
Name: count, dtype: int64

In [15]:
print(df_sample.columns.tolist())

['id', 'source_channel', 'channel_family', 'video_title', 'text', 'target', 'stance', 'tone', 'institutional', 'legitimare', 'epistemic', 'geopolitic', 'mobilizare', 'justification', 'confidence', 'fallback', 'discourse_type', 'discourse_subtype', 'type_confidence']


In [16]:
# Păstrăm doar coloanele utile și salvăm corpusul pentru etapa următoare

keep_cols = [
    "id", "text", "source_channel", "channel_family", "video_title",
    "target", "stance",
    "confidence", "discourse_type", "discourse_subtype", "type_confidence"
]

df_sample = df_sample[keep_cols].drop_duplicates(subset="text").copy()

OUT_PATH = Path("data/typed/corpus_c5_sample.jsonl")
df_sample.to_json(OUT_PATH, orient="records", lines=True, force_ascii=False)

print(df_sample.shape)
print(OUT_PATH)
df_sample.head(2)

(168, 11)
data\typed\corpus_c5_sample.jsonl


,id,text,source_channel,channel_family,video_title,target,stance,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,jigodia aia de Georgescu nu lua intrebari inca...,NicusorDanRO,,🟢 LIVE Declarații de presă susținute la Palatu...,georgescu,anti,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
10,yt_HDsCdSOtxO0_UgwZY67vyS5TxoFtknd4AaABAg,E foarte bine ca sunteți cei care cutremurați ...,RecorderRomania,,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,other_state_institution,anti,0.9,T2_grievance_anti_sistem,grievance_mobilizator,medium


## 5. Exportul bulelor finale
În această etapă transformăm corpusul tipologizat în fișiere separate, câte unul pentru fiecare bulă finală.
Codurile `T1`, `T2`, `T3`, `T4`, `T5` vin din adnotare, dar în aplicație folosim numele finale ale agenților:
| Agent | Personalitate | Cum vorbește | Ce îl definește |
|---|---|---|---|
| Personalist-salvator | devotat, admirativ, sigur | laudativ, emoțional, încrezător | vede liderul ca soluție excepțională |
| Anti-sistem | furios, suspicios, dezamăgit | acuzator, moralizator, direct | vede instituțiile și „sistemul” ca profund compromise |
| Anti-suveranist | critic, vigilent, defensiv | contestatar, mai argumentativ | respinge liderii și discursul suveranist |
| Conspiraționist | alarmist, hiper-suspicios | speculativ, revelator, totalizant | explică evenimentele prin forțe ascunse și actori externi |
| Pro-european | normativ, moderat, legalist | sobru, justificativ, procedural | apără regulile, instituțiile și ancorarea europeană |
`T6_afectiv_pozitional` nu devine agent principal în C5. Rămâne categorie transversală / rezervă.

In [17]:
BUBBLES = {
    "T1_suport_personalist": {
        "agent": "Personalist-salvator",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "T2_grievance_anti_sistem": {
        "agent": "Anti-sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "T3_opozitie_suveranista": {
        "agent": "Anti-suveranist",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "T4_conspiratie_externalism": {
        "agent": "Conspiraționist",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "T5_pro_democratic_european": {
        "agent": "Pro-european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

In [18]:
"""
OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_BUBBLE = 50

for old_type, meta in BUBBLES.items():
    bubble_df = (
        df_sample[df_sample["discourse_type"] == old_type]
        .drop_duplicates(subset="text")
        .head(N_PER_BUBBLE)
        .copy()
    )

    bubble_df["agent"] = meta["agent"]
    bubble_df["slug"] = meta["slug"]
    bubble_df["personality"] = meta["personality"]
    bubble_df["speaks"] = meta["speaks"]
    bubble_df["definition"] = meta["definition"]
    bubble_df["source_type"] = old_type

    out_path = OUT_DIR / f"{meta['slug']}.jsonl"
    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

    print(f"{meta['agent']}: {len(bubble_df)} texte -> {out_path}")

"""

'\nOUT_DIR = Path("data/bubbles")\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nN_PER_BUBBLE = 50\n\nfor old_type, meta in BUBBLES.items():\n    bubble_df = (\n        df_sample[df_sample["discourse_type"] == old_type]\n        .drop_duplicates(subset="text")\n        .head(N_PER_BUBBLE)\n        .copy()\n    )\n\n    bubble_df["agent"] = meta["agent"]\n    bubble_df["slug"] = meta["slug"]\n    bubble_df["personality"] = meta["personality"]\n    bubble_df["speaks"] = meta["speaks"]\n    bubble_df["definition"] = meta["definition"]\n    bubble_df["source_type"] = old_type\n\n    out_path = OUT_DIR / f"{meta[\'slug\']}.jsonl"\n    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)\n\n    print(f"{meta[\'agent\']}: {len(bubble_df)} texte -> {out_path}")\n\n'

## 6. Alege agentul tău și verifică textele
Fiecare membru al echipei lucrează pe un singur agent.
Datele vin din `df_sample`, iar agentul este legat de eticheta tehnică `discourse_type`.
Scopul este să alegi aproximativ 50 texte bune pentru agentul tău.

In [19]:
# Alegeți un agent și încărcați textele corespunzătoare pentru etapa următoare
AGENTS = {
    "Personalist-salvator": {
        "type": "T1_suport_personalist",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "Anti-sistem": {
        "type": "T2_grievance_anti_sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "Anti-suveranist": {
        "type": "T3_opozitie_suveranista",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "Conspiraționist": {
        "type": "T4_conspiratie_externalism",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "Pro-european": {
        "type": "T5_pro_democratic_european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
}

MY_AGENT = "Conspiraționist"  # Alegeți unul dintre agenți: "Personalist-salvator", "Anti-sistem", "Anti-suveranist", "Conspiraționist", "Pro-european"

meta = AGENTS[MY_AGENT]

my_df = (
    df_sample[df_sample["discourse_type"] == meta["type"]]
    .drop_duplicates(subset="text")
    .copy()
)

print("Agent:", MY_AGENT)
print("Tip discurs:", meta["type"])
print("Texte disponibile:", len(my_df))

my_df[["id", "type_confidence", "discourse_subtype", "text"]].head(2)

Agent: Conspiraționist
Tip discurs: T4_conspiratie_externalism
Texte disponibile: 4


,id,type_confidence,discourse_subtype,text
21,yt_PI9K4fsNHvg_UgwZ3VabzgKdxQi62W54AaABAg,medium,conspiratie_sau_externalism_slab,Sufletul Pământului în rezervorul şi motorul b...
19,yt_Z3b91sZQa5I_UgzhcgjqHRJUMUS6rTt4AaABAg,medium,conspiratie_sau_externalism_slab,Deci apare agresiunea din partea Rusiei în gen...


### Cum verifici textele
Citește textele afișate mai jos.
Dacă un text este slab, copiază ID-ul lui în lista `REMOVE_IDS`.
Elimină texte prea scurte, duplicate, ambigue sau care nu exprimă clar vocea agentului.|

- pot sa folosesc un notepad pentru IDs
daca nu gasesti 50 de comentarii bune, incarca mai multe.
- poti folosi si alte metode (export text, csv) pentru vizualizarea si alegerea comentariilor

In [20]:
for _, row in my_df.head(70).iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Confidence:", row["type_confidence"])
    print("Subtype:", row["discourse_subtype"])
    print(row["text"][:700])

ID: yt_PI9K4fsNHvg_UgwZ3VabzgKdxQi62W54AaABAg
Confidence: medium
Subtype: conspiratie_sau_externalism_slab
Sufletul Pământului în rezervorul şi motorul başinilor dracului.
ID: yt_Z3b91sZQa5I_UgzhcgjqHRJUMUS6rTt4AaABAg
Confidence: medium
Subtype: conspiratie_sau_externalism_slab
Deci apare agresiunea din partea Rusiei în general, dar nu explicit o posibilă agresiune împotriva României. Pe de altă parte, apare „independența solidară”. Nu reiese clar că trebuie să cam uităm de apărarea țării în favoarea unei solidarități militare evident în cadrul Uniunii europene? Nu este un pas în direcția federalizării fix folosind spaima comunitară de Rusia imperială?
ID: yt_XOejd9J2xqc_Ugws8_zm56ThcnowcAt4AaABAg
Confidence: high
Subtype: conspiratie_difuza
Votam masiv Nicusor Dan! Ai scris intr-o postare pe facebook ca ai fost la 7 dezbateri la care nu s-a prezentat contracandidatul. Te rog sa ne spui cum gasim si noi acele dezbateri, nu de alta dar dupa ore de cautari am constatat ca nu exista! Efec

In [21]:
# elimin textele slabe și păstrez 50

REMOVE_IDS = [
    # pune aici ID-urile textelor slabe
]

clean_df = (
    my_df[~my_df["id"].isin(REMOVE_IDS)]
    .head(50)
    .copy()
)

clean_df["agent"] = MY_AGENT
clean_df["slug"] = meta["slug"]
clean_df["personality"] = meta["personality"]
clean_df["speaks"] = meta["speaks"]
clean_df["definition"] = meta["definition"]

print("Texte finale:", len(clean_df))
clean_df[["id", "agent", "text"]].head()

Texte finale: 4


,id,agent,text
21,yt_PI9K4fsNHvg_UgwZ3VabzgKdxQi62W54AaABAg,Conspiraționist,Sufletul Pământului în rezervorul şi motorul b...
19,yt_Z3b91sZQa5I_UgzhcgjqHRJUMUS6rTt4AaABAg,Conspiraționist,Deci apare agresiunea din partea Rusiei în gen...
18,yt_XOejd9J2xqc_Ugws8_zm56ThcnowcAt4AaABAg,Conspiraționist,Votam masiv Nicusor Dan! Ai scris intr-o posta...
3,yt_sXqW2Cnw0DU_UgyFd08cuCXGNjBPclx4AaABAg,Conspiraționist,"Dictatorul Putin nu vrea pace, vrea capitulare..."


### Descrierea agentului tău
Înainte să exporți bula curată, descrie în 5–7 rânduri ce fel de voce discursivă are agentul ales.
Nu descrie o persoană reală. Descrie un tip de discurs observat în corpus.
Răspunde la aceste întrebări:
- Cum vede acest agent instituțiile, politica sau actorii publici?
- Ce ton folosește cel mai des?
- Ce tip de argumente sau acuzații apar frecvent?
- Ce îl diferențiază de celelalte bule?
- Ce ar trebui să păstreze un viitor agent AI ca să sune coerent cu această bulă?


Am ales agentul conspiraționist. Pentru acest agent, corpusul contine doar 4 comentarii, deci nu a fost posibil sa pastrez aproximativ 50 de texte. Am verificat toate exemplele disponibile si am eliminat doar textele care erau prea slabe, neclare sau nereprezentative. Au existat putine exemple, ceea ce limiteaza diversitatea agentului. Corpusul final reprezinta vocea conspirativa prin suspiciune, ideea de manipulare si explicatii bazate pe forte ascunse, dar ar avea nevoie de mai multe texte pentru etapa de retrieval.

descriere:
-
-
-

In [23]:
# export bula curată pentru etapa următoare

OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"{meta['slug']}.jsonl"

clean_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

print("Salvat:", out_path)

Salvat: data\bubbles\conspirationist.jsonl
